# Semantic Search with Gemini Embeddings + ChromaDB

Notes from working through the Vector Databases course (freeCodeCamp / codeEdx). This notebook walks through generating sentence embeddings with Google's Gemini embedding model, inspecting how similar/different sentences sit relative to each other, visualizing that in 3D, and finally storing + querying the embeddings in ChromaDB to do search by meaning instead of by keyword.

## 1. Imports

Pulling in the Gemini client plus the usual data/plotting stack.

In [5]:
from google import genai
from google.genai import types
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from IPython.display import display


## 2. Authenticate with the Gemini API

API key is pulled from Colab's secret manager so it never sits in plain text in the notebook.

In [6]:
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')

if not api_key:
    raise RuntimeError(
        "GOOGLE_API_KEY not found. "
        "Add it via Colab -> Secrets (key icon) -> enable notebook access."
    )

client = genai.Client(api_key=api_key)


## 3. Sample sentences

A small mixed bag of sentences on purpose — some near-duplicates in meaning (e.g. the two "turn off the lights" variants), some from totally different topics, and one deliberately ambiguous pair ("Apple" the company vs. "apple" the fruit) to see whether the embedding model actually separates them by context.

In [7]:
sentences = [
    "who is the founder of Pakistan?",
    "Which LLM is he best right now?",
    "What is the cpaital of Germany?",
    "What is the story of Romeo and Juilet?",
    "The Garden is beautiful.",
    "Lion is the king of the jungle.",
    "Samsung is the best company in tech",
    "Apple is my favorite fruit.",
]

EMBED_DIM = 768  # keeping this consistent everywhere below (default model output is 3072)


## 4. Generate the embeddings

Using `gemini-embedding-001`, with the output dimensionality fixed at 768 so every vector we generate later (including query vectors) stays comparable.

In [8]:
embedding_response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=sentences,
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=EMBED_DIM,
    ),
)

sentence_vectors = [e.values for e in embedding_response.embeddings]
print(f"Got {len(sentence_vectors)} vectors, each of length {len(sentence_vectors[0])}")


Got 8 vectors, each of length 768


## 5. How similar are these sentences to each other?

Building a cosine similarity matrix across all sentence pairs. Expecting the two "lights off" sentences and the two phone-related questions to cluster close together, with "Apple the fruit" sitting apart from "Apple released a new iPhone" despite sharing a word.

In [9]:
similarity_df = pd.DataFrame(
    cosine_similarity(sentence_vectors),
    index=sentences,
    columns=sentences,
)
display(similarity_df.round(2))


,who is the founder of Pakistan?,Which LLM is he best right now?,What is the cpaital of Germany?,What is the story of Romeo and Juilet?,The Garden is beautiful.,Lion is the king of the jungle.,Samsung is the best company in tech,Apple is my favorite fruit.
who is the founder of Pakistan?,1.00,0.71,0.72,0.65,0.69,0.70,0.68,0.66
Which LLM is he best right now?,0.71,1.00,0.72,0.68,0.69,0.71,0.73,0.71
What is the cpaital of Germany?,0.72,0.72,1.00,0.67,0.68,0.68,0.66,0.68
What is the story of Romeo and Juilet?,0.65,0.68,0.67,1.00,0.72,0.66,0.64,0.70
The Garden is beautiful.,0.69,0.69,0.68,0.72,1.00,0.73,0.70,0.74
Lion is the king of the jungle.,0.70,0.71,0.68,0.66,0.73,1.00,0.72,0.72
Samsung is the best company in tech,0.68,0.73,0.66,0.64,0.70,0.72,1.00,0.73
Apple is my favorite fruit.,0.66,0.71,0.68,0.70,0.74,0.72,0.73,1.00


## 6. Visualizing the embedding space

768 dimensions obviously can't be plotted directly, so PCA compresses them down to 3 components while keeping as much of the structure as possible. The goal here isn't perfect fidelity — it's to get a rough, visual sense of which sentences the model considers "close."

In [10]:
pca = PCA(n_components=3)
reduced_vectors = pca.fit_transform(np.array(sentence_vectors))

plot_df = pd.DataFrame(reduced_vectors, columns=["x", "y", "z"])
plot_df["sentence"] = sentences
plot_df


,x,y,z,sentence
0,-0.210897,-0.013975,0.138500,who is the founder of Pakistan?
1,-0.065676,-0.061745,-0.188787,Which LLM is he best right now?
2,-0.199996,0.068395,-0.058734,What is the cpaital of Germany?
3,0.084139,0.263718,-0.049939,What is the story of Romeo and Juilet?
4,0.114285,0.067413,0.123707,The Garden is beautiful.
5,0.052310,-0.100513,0.155065,Lion is the king of the jungle.
6,0.074435,-0.205902,-0.066992,Samsung is the best company in tech
7,0.151401,-0.017391,-0.052819,Apple is my favorite fruit.


In [11]:
import plotly.express as px

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    text="sentence",
    title="Sentence embeddings reduced to 3D (PCA)",
    height=800, width=1100,
)
fig.show()


## 7. Install ChromaDB

Lightweight, runs locally, persists to disk — good fit for prototyping a semantic search layer without standing up any external infrastructure.

In [12]:
!pip install -q chromadb


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 r

## 8. Store the embeddings in a Chroma collection

Each sentence is stored with its pre-computed embedding and an id so we can trace a search hit back to the original text.

In [13]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.Client(
    Settings(persist_directory="./chroma_db")
)

collection = chroma_client.get_or_create_collection(name="sentence-semantic-search")

ids = [f"sentence_{i}" for i in range(len(sentences))]

collection.add(
    ids=ids,
    embeddings=sentence_vectors,
    documents=sentences,
)

print(f"Stored {collection.count()} sentences in the '{collection.name}' collection")


Stored 8 sentences in the 'sentence-semantic-search' collection


## 9. Search by meaning, not by keyword

Querying with a word that doesn't literally appear in any stored sentence ("jungle"), to confirm the search is actually working off semantic similarity rather than string matching.

In [16]:
def semantic_search(query, n_results=2):
    query_vector = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[query],
        config=types.EmbedContentConfig(
            task_type="SEMANTIC_SIMILARITY",
            output_dimensionality=EMBED_DIM,
        ),
    ).embeddings[0].values

    return collection.query(
        query_embeddings=[query_vector],
        n_results=n_results,
    )

results = semantic_search("jungle")
for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"{distance:.4f}  ->  {doc}")


0.1107  ->  Lion is the king of the jungle.
0.1404  ->  The Garden is beautiful.


## 10. One more query, to be sure it's not a fluke

Trying a second unrelated query term ("lovely") and checking the top matches make sense.

In [20]:
results = semantic_search("lovely")
for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"{distance:.4f}  ->  {doc}")


0.0897  ->  The Garden is beautiful.
0.1540  ->  Apple is my favorite fruit.


## Takeaways

- Embeddings place semantically related sentences close together in vector space, even with zero word overlap.
- Keeping `output_dimensionality` consistent between stored vectors and query vectors matters — mismatched dimensions is an easy bug to hit.
- PCA is useful for building intuition about the embedding space, but it's a lossy 2D/3D approximation of a much higher-dimensional reality — don't over-read the plot.
- ChromaDB made it trivial to go from "a list of vectors" to "a queryable semantic search index" with very little setup, which is a good building block for a RAG pipeline.